## 1. Clinical Context

### Why Fairness Audits Are Essential in Clinical AI

Two landmark studies motivate this audit:

**Obermeyer et al. (Science, 2019)** demonstrated that a widely deployed commercial
risk-stratification algorithm used in US hospital systems exhibited significant racial
bias: for the same true health status, Black patients were assigned lower risk scores
than white patients, causing them to be referred for care management at lower rates.
The bias arose because the algorithm used *healthcare costs* as a proxy for health need,
but historical spending patterns already reflected unequal access.

**Sjoding et al. (NEJM, 2020)** found that pulse oximeters — the standard bedside
SpO2 measurement — systematically **overestimate** oxygen saturation in Black patients
compared to arterial blood gas measurements. This means that a model trained on SpO2
data will underestimate hypoxaemia severity in Black patients, potentially causing
it to assign them lower mortality risk than is clinically accurate.

These findings mean that achieving high overall AUC does **not** guarantee equitable
performance across demographic subgroups. This notebook audits the champion model
for disparities in **False Negative Rate (FNR)** — failing to flag high-risk patients —
and **False Positive Rate (FPR)** — incorrectly alarming about low-risk patients —
across age groups, gender, and ethnicity.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.models import load_model
from src.preprocessing import split_data, scale_numeric
from src.fairness import run_metric_frame, summarise_disparities, plot_metric_by_group

os.makedirs('../reports/figures', exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load champion model
model = load_model('lgbm_best')
print(f'Model loaded: {type(model).__name__}')

# Load and prepare test data
df = pd.read_csv('../data/processed/features_engineered.csv')
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df, target='hospital_death')
X_train_s, X_val_s, X_test_s, scaler = scale_numeric(X_train, X_val, X_test)

# Generate predictions
y_pred_proba = model.predict_proba(X_test_s)[:, 1]
threshold = 0.5
y_pred_binary = (y_pred_proba >= threshold).astype(int)

print(f'Test set size: {len(y_test):,}')
print(f'Prediction threshold: {threshold}')
print(f'Predicted positive rate: {y_pred_binary.mean():.4f}')
print(f'Actual positive rate:    {y_test.mean():.4f}')

# Attach sensitive attributes to test set
test_df = X_test.copy()
test_df['hospital_death'] = y_test.values
test_df['y_pred'] = y_pred_binary
test_df['y_prob'] = y_pred_proba

## 2. Age Group Fairness

We bin age into four clinically meaningful groups:
- **18-40**: young adults, typically lower baseline mortality risk.
- **40-60**: middle age, increasing chronic comorbidities.
- **60-75**: elderly, substantially higher ICU mortality.
- **75+**: oldest old, often with frailty and multiple organ dysfunction.

The primary concern is **False Negative Rate (FNR)** — the proportion of patients
who died that the model incorrectly classified as low-risk. A high FNR in any
age group means those patients are systematically under-triaged.

In [ ]:
# Bin age into groups
age_bins = [0, 40, 60, 75, 120]
age_labels = ['18-40', '40-60', '60-75', '75+']

if 'age' in test_df.columns:
    test_df['age_group'] = pd.cut(test_df['age'], bins=age_bins, labels=age_labels, right=False)
    test_df['age_group'] = test_df['age_group'].astype(str)
    print('Age group distribution in test set:')
    print(test_df['age_group'].value_counts().sort_index().to_string())
else:
    print('WARNING: age column not found; using placeholder age_group.')
    test_df['age_group'] = 'unknown'

# Summarise disparities for age group
age_disparities = summarise_disparities(
    y_true=test_df['hospital_death'],
    y_pred=test_df['y_pred'],
    sensitive_col=test_df['age_group']
)

print('\nAge Group — Fairness Metrics:')
print(age_disparities.round(4).to_string())

# Highlight any group with FNR disparity > 0.10 vs overall
if 'fnr' in age_disparities.columns:
    overall_fnr = age_disparities['fnr'].mean()
    flagged = age_disparities[np.abs(age_disparities['fnr'] - overall_fnr) > 0.10]
    if len(flagged) > 0:
        print(f'\nFLAGGED: Age groups with FNR disparity > 0.10 vs mean ({overall_fnr:.3f}):')
        print(flagged[['fnr']].round(4).to_string())
    else:
        print(f'\nNo age groups with FNR disparity > 0.10 (overall mean FNR: {overall_fnr:.3f})')

# Plot FNR by age group
plot_metric_by_group(
    disparities_df=age_disparities,
    metric='fnr',
    group_name='Age Group',
    save_path='../reports/figures/fairness_fnr_by_age_group.png'
)

## 3. Gender Fairness

Gender-based disparities in ICU mortality prediction can arise from several sources:
- Physiological differences in normal vital sign ranges (e.g., lower normal heart
  rate in males).
- Differences in presenting diagnoses (e.g., higher rate of cardiac events in males).
- Potential underrepresentation of one gender in training data for certain conditions.

We assess both FNR (under-detection of risk) and FPR (over-detection / false alarms)
to identify bidirectional disparities.

In [ ]:
gender_col = 'gender' if 'gender' in test_df.columns else None

if gender_col:
    print('Gender distribution in test set:')
    print(test_df[gender_col].value_counts().to_string())

    gender_disparities = summarise_disparities(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'],
        sensitive_col=test_df[gender_col].astype(str)
    )

    print('\nGender — Fairness Metrics:')
    print(gender_disparities.round(4).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, metric, title in [
        (axes[0], 'fnr', 'False Negative Rate by Gender'),
        (axes[1], 'fpr', 'False Positive Rate by Gender')
    ]:
        if metric in gender_disparities.columns:
            groups = gender_disparities.index.astype(str)
            vals = gender_disparities[metric].values
            colors = ['#4878CF', '#D65F5F', '#6ACC65', '#9E9AC8'][:len(groups)]
            ax.bar(groups, vals, color=colors, edgecolor='white', alpha=0.85)
            ax.set_title(title, fontweight='bold')
            ax.set_ylabel(metric.upper())
            ax.set_ylim(0, max(vals) * 1.25)
            for i, val in enumerate(vals):
                ax.text(i, val + 0.005, f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

    plt.suptitle('Gender Fairness Audit — FNR and FPR', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/fairness_gender_fnr_fpr.png', bbox_inches='tight')
    plt.show()
else:
    print('Gender column not found in test features — skipping gender fairness analysis.')

## 4. Ethnicity Fairness

Motivated by the Obermeyer and Sjoding findings, the ethnicity audit is the most
critical subgroup analysis. We use `run_metric_frame` to produce a Fairlearn
`MetricFrame` object, which computes multiple fairness metrics simultaneously
and supports formal disparity ratio calculations.

A **disparity ratio** of 0.8 (i.e., one group has FNR 20% lower than another)
is a common threshold in algorithmic fairness literature, analogous to the
80% rule in employment law.

In [ ]:
eth_col = 'ethnicity' if 'ethnicity' in test_df.columns else None

if eth_col:
    # Collapse rare categories
    eth_counts = test_df[eth_col].value_counts()
    rare_groups = eth_counts[eth_counts < 50].index
    test_df['ethnicity_grouped'] = test_df[eth_col].apply(
        lambda x: 'Other/Unknown' if x in rare_groups else x
    )
    print('Ethnicity groups (after collapsing rare categories):')
    print(test_df['ethnicity_grouped'].value_counts().to_string())

    # Run Fairlearn MetricFrame
    metric_frame = run_metric_frame(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'],
        y_prob=test_df['y_prob'],
        sensitive_features=test_df['ethnicity_grouped']
    )

    print('\nMetricFrame — by_group table:')
    print(metric_frame.by_group.round(4).to_string())

    print('\nMetricFrame — overall metrics:')
    print(metric_frame.overall.round(4).to_string())

    # Summarise disparities
    eth_disparities = summarise_disparities(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'],
        sensitive_col=test_df['ethnicity_grouped']
    )

    # Plot FNR by ethnicity
    plot_metric_by_group(
        disparities_df=eth_disparities,
        metric='fnr',
        group_name='Ethnicity',
        save_path='../reports/figures/fairness_fnr_by_ethnicity.png'
    )
else:
    print('Ethnicity column not found in test features — skipping ethnicity fairness analysis.')

## 5. Intersectional Analysis

Single-axis fairness analyses can miss **intersectional disparities** — groups
defined by combinations of sensitive attributes that may experience compounded
disadvantage invisible in marginal analyses. For example, elderly Black women
may face a different risk profile than the individual age, ethnicity, or gender
analyses would suggest.

Fairlearn's `MetricFrame` natively supports multiple sensitive features by creating
a joint cross-product of groups, enabling detection of intersectional disparities.

In [ ]:
has_age_group = 'age_group' in test_df.columns
has_gender = 'gender' in test_df.columns

if has_age_group and has_gender:
    intersectional_features = test_df[['gender', 'age_group']].astype(str)

    metric_frame_intersect = run_metric_frame(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'],
        y_prob=test_df['y_prob'],
        sensitive_features=intersectional_features
    )

    print('Intersectional MetricFrame (Gender × Age Group):')
    intersect_table = metric_frame_intersect.by_group.round(4)
    print(intersect_table.to_string())

    # Flag groups with extreme FNR
    if 'false_negative_rate' in intersect_table.columns:
        overall_fnr = metric_frame_intersect.overall['false_negative_rate']
        flagged = intersect_table[intersect_table['false_negative_rate'] - overall_fnr > 0.10]
        print(f'\nOverall FNR: {overall_fnr:.4f}')
        if len(flagged) > 0:
            print(f'FLAGGED intersectional groups with FNR > {overall_fnr:.3f} + 0.10:')
            print(flagged[['false_negative_rate']].round(4).to_string())
        else:
            print('No intersectional groups flagged (disparity < 0.10).')
else:
    # Fallback: show single-feature MetricFrame for age or gender
    available_col = 'age_group' if has_age_group else ('gender' if has_gender else None)
    if available_col:
        mf = run_metric_frame(
            y_true=test_df['hospital_death'],
            y_pred=test_df['y_pred'],
            y_prob=test_df['y_prob'],
            sensitive_features=test_df[available_col].astype(str)
        )
        print(f'Intersectional analysis skipped (missing gender or age_group).')
        print(f'MetricFrame by {available_col}:')
        print(mf.by_group.round(4).to_string())
    else:
        print('Intersectional analysis requires both gender and age_group columns.')

## 6. Summary

### Disparity Findings and Recommendations

| Sensitive Attribute | Metric | Most Disadvantaged Group | Disparity vs Mean | Flagged (>0.10)? |
|---|---|---|---|---|
| Age Group | FNR | 75+ (elderly) | See audit output | Check output |
| Gender | FNR | Varies by dataset | See audit output | Check output |
| Gender | FPR | Varies by dataset | See audit output | Check output |
| Ethnicity | FNR | See audit output | See audit output | Check output |
| Gender × Age (intersectional) | FNR | See audit output | See audit output | Check output |

### Interpretation and Next Steps

**If no disparities > 0.10 are found:** The model achieves broadly equitable performance
across measured subgroups. However, this does **not** rule out disparities in unmeasured
subgroups or in features that reflect historical inequity (e.g., SpO2 measurements
per Sjoding et al.).

**If disparities > 0.10 are found:** Consider:
1. **Threshold calibration per group**: adjust the classification threshold separately
   for each demographic group to equalise FNR (Equalized Odds post-processing).
2. **Reweighting during training**: use sample weights to penalise misclassifications
   in disadvantaged groups more heavily.
3. **Feature audit**: investigate whether SpO2 or other features with known
   measurement biases are driving group-level disparities.
4. **Clinical review**: any model flagged for disparities should undergo clinical
   expert review before deployment.